# NEM price forecasting: fine-tune Chronos-2 and run the benchmark

Benchmarks time-series foundation models (Amazon Chronos-2) against baselines on Victorian (VIC1) electricity prices
from AEMO's public data, then fine-tunes Chronos-2 on 2023-24 and tests everything on 2025 onward.

This notebook clones the repo, installs it, **fine-tunes Chronos-2 on the GPU**, runs the backtest and shows the results.

**Before you start:** Runtime > Change runtime type > **GPU** (a T4 is fine). Then edit the settings cell and Run all.

What it does:
1. Clone the repo and install it (keeps Colab's own CUDA PyTorch).
2. Check a GPU is present (it stops otherwise, so you do not spend hours on a CPU by accident).
3. `make test`: quick unit tests.
4. `make finetune`: LoRA fine-tune on 2023-24 prices only (the test period is never used for training).
5. `make backtest`: each model forecasts the next 48 half-hours from every midnight since 1 Jan 2025.
6. Show the results table and the significance table.

Data: AEMO's public NEM data, committed in the repo under `datasets/` (see `datasets/README.md` to rebuild it with nemosis).

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/janxl/nem-timeseries.git"   # a private repo needs a token in the URL, or make it public
BRANCH = "main"

# Fine-tuning (defaults are untuned; change them freely)
NUM_STEPS = 1000
BATCH_SIZE = 32
LEARNING_RATE = 1e-5

# Models to score. The ablation variants (chronos2_zs_ctx3d, ...) are available too but slower.
MODELS = "naive_yesterday,naive_last_week,lightgbm,chronos2_zs,chronos2_zs_cal,chronos2_ft"

ALLOW_CPU = False        # leave False: a CPU fine-tune takes hours
SAVE_TO_DRIVE = False    # True: copy the checkpoint and results to Google Drive at the end

## 2. Get the code and install

In [ ]:
import os

REPO_DIR = REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]   # e.g. nem-timeseries
if os.path.basename(os.getcwd()) != REPO_DIR:    # safe to re-run this cell
    if not os.path.isdir(REPO_DIR):
        !git clone --branch {BRANCH} {REPO_URL}
    %cd {REPO_DIR}
!git pull --ff-only

In [ ]:
# Colab already has a CUDA build of PyTorch; the requirement torch>=2.3 is met, so it is kept.
!pip install -q -e ".[models,dev]"
import sys
sys.path.insert(0, os.path.abspath("src"))   # so this session can import the package without a restart

## 3. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no NVIDIA GPU found"
from nem_ts.device import get_device

device = get_device()
print("device:", device)
if device == "cpu" and not ALLOW_CPU:
    raise RuntimeError("No GPU. Use Runtime > Change runtime type > GPU, or set ALLOW_CPU = True (slow).")

## 4. Unit tests

In [ ]:
# RUN= (empty) runs the pip-installed commands directly; on a laptop the Makefile uses uv instead.
!make test RUN=

## 5. Fine-tune Chronos-2

LoRA fine-tune on VIC1 prices from 2023-01-01 to 2024-12-31 only, with a 14-day context, forecasting 48 half-hours.
The checkpoint is saved to `models/chronos2_ft/`. The backtest refuses to score any forecast origin the model trained on.

In [ ]:
!make finetune RUN= ARGS="--num-steps {NUM_STEPS} --batch-size {BATCH_SIZE} --learning-rate {LEARNING_RATE}"

In [ ]:
import json
print(json.dumps(json.load(open("models/chronos2_ft/meta.json")), indent=2))

## 6. Backtest

Every model forecasts the next 48 half-hours (24 hours) from each NEM midnight, 1 Jan 2025 onward. Predictions are saved
under `results/VIC1/`, so a re-run reuses them (add `--force` to recompute).

In [ ]:
!make backtest RUN= ARGS="--models {MODELS}"

## 7. Results

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("results/VIC1/metrics.md").read()))

How to read it: MAE, RMSE and pinball are in $/MWh (lower is better). *MAE spikes* uses the top 5% of prices; *P10-P90
coverage* should be near 80%. The second table compares every model with `chronos2_zs` day by day; a `*` means the
difference is unlikely to be luck. The question this notebook answers: does fine-tuning (`chronos2_ft`) beat zero-shot?

## 8. Keep the results (optional)

In [ ]:
import shutil

shutil.make_archive("nem_ts_results", "zip", ".", "results")   # metrics and predictions
print("wrote nem_ts_results.zip")

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/nem-time-series
    !cp -r models results /content/drive/MyDrive/nem-time-series/
    print("copied models/ and results/ to Google Drive: nem-time-series/")

In [ ]:
# Download the results zip to your computer (Colab only).
from google.colab import files

files.download("nem_ts_results.zip")